In [1]:
# Cell 1: Setup
%load_ext autoreload
%autoreload 2

from datetime import datetime
import audit_engine as ae
import pandas as pd
pd.options.mode.chained_assignment = None
pd.set_option('display.max_columns', None)
import time
import concurrent.futures
from tqdm.auto import tqdm
import os
from datetime import date,timedelta
import duckdb
import ast
import numpy as np

In [2]:
# import importlib
# import audit_engine as ae

# # 1. Reload the specific sub-module where the logic changed
# importlib.reload(ae.fs_api)
# importlib.reload(ae.config)
# importlib.reload(ae.core)
# importlib.reload(ae.utils)
# importlib.reload(ae.db_logic)
# importlib.reload(ae.sql_queries)
# importlib.reload(ae)

# # 2. Reload the main package to refresh the __init__ mappings
# importlib.reload(ae)

# print("all libs reimported and ae namespace updated.")

# import gc
# import pandas as pd

# # This clears up unreferenced objects/file handles
# gc.collect()

In [3]:
ts_prefix = datetime.now().strftime("%Y%m%d%H%M")
fn = f'data_reports/tickets_{ts_prefix}.json'
conn = ae.get_sf_connection()

Initiating login request with your identity provider. Press CTRL+C to abort and try again...
Going to open: https://login.microsoftonline.com/ef6f731e-41f8-43bd-9df6-8982a75c2f24/saml2?SAMLRequest=lZJdb9owFIb%2FSuRd58NJRhMLqCiIFY1uDOg0cWdih1o4durjEPj3c0KRuotW2l3kPK%2F92O8Z3p8r6Z24AaHVCOEgQh5XhWZCHUboeTv3M%2BSBpYpRqRUfoQsHdD8eAq1kTSaNfVFr%2FtpwsJ7bSAHpf4xQYxTRFAQQRSsOxBZkM3lakjiISG201YWW6F3k8wQF4MY6w1uEgXB6L9bWJAzbtg3aJNDmEMZRFIVRHjqqQ77c%2BLO70wc8DqO04x3h8NWb24NQ1yf4TGt%2FhYA8brcrf%2FVzs0Xe5KY61QqaipsNNydR8Of18ioAzqC6UDzIUxw04HMK1scBKN2Wkh55oau6sW7bwH2FJWeh1Afhbr6YjVB9FAyS4684Siv2iOmfMz7N9g%2F1vPimsrJeT%2FV%2BmajdISl26vtrViDv963auKt2AdDwheoKtW4pigd%2BlPh4sMUpSXOS4CDNsx3yZq5Qoajtkzfr3iOoRGE06NJqJYXivSUvB%2BVdgrmf4jLz02TP%2FJyVAz%2FLs5jefS3iMk7DruYYXUeH9CJm%2FN8PMgzfx9%2FG8IdrZjFbaSmKizfXpqL24%2BJwgPsVwfyyRwmvqJATxgwHcAVKqdup4dS6abem4SgcX0%2F9d97HfwE%3D&RelayState=ver%3A3-hint%3A27309133419580866-ETMsDgAAAZz3H6JqABRBRVMvQ0JDL1BLQ1M1UGFkZGluZwEAABAAEEX3LOQ8l9ZaDibHZtvMDuMAAACgW%2BVzN%2B7F

In [4]:
print(f"Started at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# 1. Fetch tickets using the engine
# Note: Ensure start_date and end_date are valid strings for normalize_date
# 1. Get Input
raw_input = input('Enter date (yyyy-mm-dd) or days back (max 60): ').strip()

# 2. Define our Safety Limit
MAX_LOOKBACK = 60
default_date = date.today() - timedelta(days=20)

# 3. Logic Flow
if not raw_input:
    final_date = default_date

elif raw_input.isdigit():
    days = int(raw_input)
    # Cap the days at 60
    days = min(days, MAX_LOOKBACK)
    final_date = date.today() - timedelta(days=days)

else:
    try:
        # Try to parse the specific date string
        parsed_date = datetime.strptime(raw_input, '%Y-%m-%d').date()
        
        # Check if the entered date is older than 60 days
        earliest_allowed = date.today() - timedelta(days=MAX_LOOKBACK)
        
        if parsed_date < earliest_allowed:
            print(f"Date too old! Capping at {MAX_LOOKBACK} days ago.")
            final_date = earliest_allowed
        else:
            final_date = parsed_date
            
    except ValueError:
        # This catches "abcde" or "2023-13-45"
        print("Invalid input detected. Defaulting to 20 days ago.")
        final_date = default_date

sd1 = final_date.strftime('%Y-%m-%d')
print(f"Final Start Date: {sd1}")

ed1 = input(prompt='enter end date in format yyyy-mm-dd : ').strip()
if not ed1:
    ed1 = date.today().strftime('%Y-%m-%d')
print(f"Start Date set to: {sd1}")
print(f"End Date set to: {ed1}")

sd2 = sd1+'T00:00:00Z'
ed2 = ed1+'T23:59:59Z'

# tickets = ae.get_tickets_between_dates(
#     start_date = "2026-01-15T00:00:00Z",
#     end_date = "2026-01-27T23:59:59Z"
# )
tickets = ae.get_tickets_between_dates(
    start_date = sd2,
    end_date = ed2
)

if 1==1:
    ae.fs_api.save_or_load_tickets(
        filename=fn,
        tickets=tickets,
        ts_prefix=ts_prefix,
        mode="save"
    )
    
    # 3. Load tickets back for verification
    tickets = ae.fs_api.save_or_load_tickets(
        filename=fn,
        mode="load"
    )
print(f"\n Total tickets loaded: {len(tickets)}")
print(f"Ended at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

Started at 2026-03-16 20:19:36


Enter date (yyyy-mm-dd) or days back (max 60):  31


Final Start Date: 2026-02-13


enter end date in format yyyy-mm-dd :  


Start Date set to: 2026-02-13
End Date set to: 2026-03-16

 Total tickets loaded: 2637
Ended at 2026-03-16 20:20:25


In [5]:
# --- YOUR CONTROLS START ---
time.sleep(65)
target_tickets = []

for t in tickets:
    ticko_subject = t.get('subject', '').lower().replace(' ', '')
    cf = t.get('custom_fields', {})
    audit_status = cf.get('audit_status_1')
    cleaned_subject = ticko_subject.lower().replace(' ', '')
    is_price_change = ("pricechange" in cleaned_subject and "external" not in cleaned_subject)
    audit_status_check = bool(audit_status and audit_status.strip().lower() == 'new')
    # audit_status_check = True
    
    if is_price_change and audit_status and audit_status_check:
        target_tickets.append(t)
# --- YOUR CONTROLS END ---

print(f"Filtered down to {len(target_tickets)} tickets. Starting parallel run...")

# 1. Start Timing
start_time = datetime.now()

# 2. Parallel Extraction (Using the list you just built)
price_change_df = ae.process_tickets_parallel(target_tickets, max_workers=10)
price_change_df = ae.build_price_change_df(df_rows=price_change_df.to_dict('records'), ts_prefix=ts_prefix,save_excel=False)

# 3. End Timing
print(f"Total Duration: {datetime.now() - start_time}")

Filtered down to 28 tickets. Starting parallel run...
Starting parallel extraction for 28 tickets...


API Extraction:   0%|          | 0/28 [00:00<?, ?it/s]

Total Duration: 0:00:03.824157


In [6]:
# ae.fs_api.view_ticket(122629,False)

In [7]:
# Calculate counts and basic checks
price_change_df['ticko_pcw_worksheet_details_ct'] = price_change_df['ticko_pcw_worksheet_details'].apply(lambda x: len(x) if x is not None else 0)
# Validation: Ensure details count is numeric
price_change_df['worksheet_checks'] = pd.to_numeric(price_change_df["ticko_pcw_worksheet_details_ct"], errors="coerce").notnull()

print('df_rows length: ', len(price_change_df.to_dict('records')))
print('price_change_df shape: ', price_change_df.shape)

# 4. Reorder and Rename using Config
# We pull 'price_change_df_col_order' and 'price_change_df_rename_dict' from ae.config
price_change_df_send = ae.utils.reorder_and_rename_columns(
    price_change_df, 
    ae.config.price_change_df_col_order, 
    ae.config.price_change_df_rename_dict
)

# 5. Business Logic Formatting
price_change_df_send['attachment_ready'] = 0
price_change_df_send['old_unique_key_list'] = 0

# Map Markdowns using your config dict
price_change_df_send['Markdown_Code'] = (
    price_change_df_send['Markdown']
    .map(ae.config.mkdown_dict)
    .fillna(99)
    .astype('Int64')
)

# Fix Dates
price_change_df_send['Effective_Date_fixed'] = pd.to_datetime(price_change_df_send['Effective_Date2']).dt.date
price_change_df_send['Ticket_Creation_DT'] = pd.to_datetime(price_change_df_send['Ticket_Creation_Datetime']).dt.date
price_change_df_send['Worksheet_Details'] = price_change_df_send['Worksheet_Details'].astype(str)
price_change_df_send['Percent_Off2'] = price_change_df_send['Percent_Off'].astype(str).str.extract(r'(\d+)')[0].astype(float)
price_change_df_send['Percent_Off2'] = price_change_df_send['Percent_Off2'].apply(ae.utils.normalize_pct)

# 6. Save the Audit Draft
output_fn = f'data_reports/price_change_df_send_1_{ts_prefix}.xlsx'
price_change_df_send.to_excel(output_fn, index=False)
print(f"Audit file saved: {output_fn}")

df_rows length:  28
price_change_df shape:  (28, 33)
Audit file saved: data_reports/price_change_df_send_1_202603162019.xlsx


In [8]:
# price_change_df_send['Po_Agent']

In [9]:
time.sleep(65)
ae.utils.delete_all_files(ae.config.down_path)

break_limit = 9000

print(f"Starting downloads for {len(price_change_df_send)} potential attachments...")
# Prepare tasks from the dataframe
tasks = [
    (row["canonical_last_part"], row["unique_key"]) 
    for _, row in price_change_df_send.iterrows()
    if pd.notna(row["canonical_last_part"]) and len(str(row["canonical_last_part"])) > 8
]

# Run the bulk download
ae.fs_api.bulk_download_attachments_v2(tasks, max_workers=5)

print(f"\nFinished. Files are in: {ae.config.down_path}")

Starting downloads for 28 potential attachments...
Initiating parallel download with 5 workers...


Downloading:   0%|          | 0/28 [00:00<?, ?it/s]


Finished. Files are in: C:\Users\MBathija\BI_Engineering\FreshAuditv2\Down_Folder


In [10]:
ae.utils.delete_all_files("Fixed_attachments")

attachment_df_list = []
issue_df_list = []
break_limit = 5000

# 2. Main Processing Loop
for enm, (idx, can_id, uk, tkt, tkt_dt,agt) in enumerate(zip(
    price_change_df_send.index,
    price_change_df_send["canonical_last_part"], 
    price_change_df_send["unique_key"],
    price_change_df_send['Ticket_Id'],
    price_change_df_send['Ticket_Creation_DT'],
    price_change_df_send['Po_Agent']
)):
    
    # Pre-filtering logic
    cid = str(can_id).strip()
    if pd.isna(can_id) or cid.lower() == 'nan' or len(cid) <= 5:
        continue
    if enm >= break_limit:
        break
        
    # Construct paths using ae.config
    file_path = os.path.join(ae.config.down_path, uk if uk.lower().endswith(".xlsx") else f"{uk}.xlsx")
    
    if not os.path.exists(file_path):
        continue

    try:
        # Use the reader from your core logic
        # Assuming COL_MAP and REQUIRED_SETS are now in ae.config or ae.core
        temp1, missing_cols = ae.core.read_price_change_sheet_v4(
            file_path, 
            ae.config.COL_MAP, 
            ae.config.REQUIRED_SETS
        )
        
        print(f"Processing: {enm}", end="\r")

        if temp1 is None:
            issue_df_list.append([uk, tkt,agt, tkt_dt, file_path, f"Schema mismatch: {missing_cols}"])
            continue

        # Data Cleaning
        temp1 = temp1.replace(r'^\s*$', pd.NA, regex=True).dropna(how="all").reset_index(drop=True)
        req_cols_temp1 = list(ae.config.req_cols)
        temp1 = temp1[req_cols_temp1]
        # Remove "Unnamed" columns
        unnamed = [c for c in temp1.columns if str(c).lower().startswith("unnamed")]
        temp1 = temp1.drop(columns=unnamed, errors="ignore")
        temp1 = temp1.loc[:, temp1.columns.notna() & (temp1.columns.astype(str).str.strip() != "")]

        # SKU Logic using SKU_COL_SETS from config
        unique_skus = None
        for cols in ae.config.SKU_COL_SETS:
            if all(c in temp1.columns for c in cols):
                unique_skus = temp1[cols].drop_duplicates().shape[0]
                break
        
        temp1['unique_skus'] = unique_skus
        temp1['uk'] = uk
        attachment_df_list.append(temp1)

        # Save to "Fixed" directory using path from config
        fixed_path = os.path.join(ae.config.attachment_path, f"fixed_{os.path.basename(file_path)}")
        temp1.to_excel(fixed_path, index=False)
        
        # Update your main tracking DF
        price_change_df_send.loc[idx, 'attachment_ready'] = 1

    except Exception as e:
        issue_df_list.append([uk, tkt,agt, tkt_dt, file_path, str(e)])
        continue

print(f"\nFinished. Successes: {len(attachment_df_list)} | Issues: {len(issue_df_list)}")

Processing: 27
Finished. Successes: 21 | Issues: 6


In [11]:
issue_files = pd.DataFrame(issue_df_list,columns=['file_id','ticket','po_agent','ticket_create_date','path','error'])
print('issue templates are : '+str(issue_files.shape[0]))
print('correct templates are : '+str(price_change_df_send[price_change_df_send['attachment_ready']==1].shape[0]))
print('issue rate is : '+str(round(issue_files.shape[0]/price_change_df_send.shape[0],2)*100)+'%')
issue_files['size_kb'] = issue_files['path'].apply(lambda x: round(os.path.getsize(x) / 1024, 2) if os.path.exists(x) else 0)
issue_files['missing_count'] = issue_files['error'].apply(ae.utils.get_count)
issue_files.sort_values(by=['missing_count'],inplace=True)

issue templates are : 6
correct templates are : 21
issue rate is : 21.0%


In [12]:
# 1. First loop: Merge attachments
dub_can_tickets = price_change_df_send[(price_change_df_send['canonical_url_ct'] > 1) & (price_change_df_send['attachment_ready'] == 1)]['Ticket_Id'].unique().tolist()

if len(dub_can_tickets) > 0:
    print(dub_can_tickets)
    for i in dub_can_tickets:
        # 1. Reset the list for every new ticket
        current_ticket_dfs = []
        
        # Get UKs for this specific ticket
        mask = (price_change_df_send['Ticket_Id'] == i) & (price_change_df_send['attachment_ready'] == 1)
        uks = price_change_df_send[mask]['unique_key'].unique().tolist()
        
        if len(uks) > 1:
            for uk in uks:
                # Pointing to your package config path
                fixed_path = os.path.join(ae.config.attachment_path, f"fixed_{uk}.xlsx")
                
                if os.path.exists(fixed_path):
                    attach = pd.read_excel(fixed_path)
                    attach['orig_file'] = uk
                    current_ticket_dfs.append(attach)
    
            # 2. Combine and handle the result for this specific ticket
            if current_ticket_dfs:
                combined_for_i = pd.concat(current_ticket_dfs, ignore_index=True)
                # Note: uk here will be the last one from the loop above
                merged_filename = f"fixed_{uk}_merged.xlsx"
                new_path = os.path.join(ae.config.attachment_path, merged_filename)
                
                combined_for_i.to_excel(new_path, index=False) # index=False usually cleaner for Excel
                temp_name = merged_filename.replace('.xlsx','').replace('fixed_','')
                price_change_df_send.loc[price_change_df_send['Ticket_Id'] == i, 'unique_key'] = temp_name
                uks_string = ae.utils.list_to_str(uks)
                price_change_df_send.loc[price_change_df_send['Ticket_Id'] == i, 'old_unique_key_list'] = uks_string

# 2. Second loop: Deduplicate and format
if len(dub_can_tickets) > 0:
    for i in dub_can_tickets:
        ignore_cols = ['tickt_canonical_url', 'canonical_last_part','old_unique_key_list'] 
        cols_to_use = [c for c in price_change_df_send.columns if c not in ignore_cols]
        
        price_change_df_send['Worksheet_Details'] = price_change_df_send['Worksheet_Details'].astype(str)
        price_change_df_send = price_change_df_send.drop_duplicates(subset=cols_to_use).reset_index(drop=True)
        
        # Using the utility from your package
        price_change_df_send['Worksheet_Details'] = price_change_df_send['Worksheet_Details'].apply(ae.utils.string_to_list_quoted)

# 3. Final Export
price_change_df_send['Worksheet_Details'] = price_change_df_send['Worksheet_Details'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
price_change_df_send['Cost_Updates_req_flag'] = price_change_df_send['Cost_Updates'].str.lower().str.strip().str.startswith('yes')
price_change_df_send['Worksheet_Details_Cleaned'] = (price_change_df_send['Worksheet_Details'].apply(ae.utils.clean_row_ws))
price_change_df_send['Worksheet_Details_Cleaned_PCW'] = price_change_df_send['Worksheet_Details_Cleaned'].apply(lambda ws_list: [ws for ws in ws_list if int(ws) < ae.config.worksheet_anchor])
price_change_df_send['Worksheet_Details_Cleaned_CS'] = price_change_df_send.apply(lambda row: [ws for ws in row['Worksheet_Details_Cleaned'] if int(ws) >= ae.config.worksheet_anchor and row['Cost_Updates_req_flag']], axis=1)
price_change_df_send.to_excel(f'data_reports/price_change_df_send_2_{ts_prefix}.xlsx', index=False)
price_change_df_send.to_pickle(f'data_reports/price_change_df_send_2_{ts_prefix}.pkl')

In [13]:
# import importlib
# import audit_engine as ae

# # 1. Reload the specific sub-module where the logic changed
# importlib.reload(ae.fs_api)
# importlib.reload(ae.config)
# importlib.reload(ae.core)
# importlib.reload(ae.utils)
# importlib.reload(ae.db_logic)
# importlib.reload(ae.sql_queries)
# importlib.reload(ae)

# # 2. Reload the main package to refresh the __init__ mappings
# importlib.reload(ae)

# print("all libs reimported and ae namespace updated.")

# import gc
# import pandas as pd

# # This clears up unreferenced objects/file handles
# gc.collect()

In [14]:
# --- DEBUG TOGGLE ---
# Set this to None to process everything, or a specific ID to isolate one ticket
target_ticket_id = None
# target_ticket_id = 119628

# 1. Clean the Price Change Type
price_type_clean = price_change_df_send['Price_Change_Type'].astype(str).str.strip().str.lower()
valid_types = ['perm markdown', 'regular / regular', 'markdown cancel']

# 2. Build the conditions
condition_types = price_type_clean.isin(valid_types)
condition_attachment = price_change_df_send['attachment_ready'] == 1

# 3. Handle the single ticket logic
if target_ticket_id is not None:
    condition_ticket = price_change_df_send['Ticket_Id'] == target_ticket_id
else:
    condition_ticket = True # This acts as a "pass-through" for all rows

# 4. Combine and Apply
mask = condition_types & condition_attachment & condition_ticket

# # 4. Apply the filter
price_change_df_send = price_change_df_send[mask].copy().reset_index(drop=True)
price_change_df_send.to_excel(f'data_reports/price_change_df_send_3_{ts_prefix}.xlsx', index=False)
price_change_df_send2 = duckdb.query(ae.pcdf_q).to_df()

ws_list_pcw = (
    price_change_df_send2['Worksheet_Details_Cleaned_PCW']
    .explode()
    .dropna()
    .astype(str)
    .str.strip()
    # 1. Ensure it only contains digits (0-9)
    # 2. Ensure the length is greater than 3
    .loc[lambda x: (x.str.isdigit()) & (x.str.len() > 3)]
    .unique()
    .tolist()
)

ws_list_pcw_2 = ae.utils.list_to_quoted_str_v2(ws_list_pcw)

ws_list_cs = (
    price_change_df_send2['Worksheet_Details_Cleaned_CS']
    .explode()
    .dropna()
    .astype(str)
    .str.strip()
    # 1. Ensure it only contains digits (0-9)
    # 2. Ensure the length is greater than 3
    .loc[lambda x: (x.str.isdigit()) & (x.str.len() > 3)]
    .unique()
    .tolist()
)

ws_list_cs_2 = ae.utils.list_to_quoted_str_v2(ws_list_cs)

sql_queryf = ae.sql_queries.sql_query_1.replace('{wd_string}', ws_list_pcw_2)
sql_df = ae.run_query(conn, sql_queryf)

sql_queryf2 = ae.sql_queries.sql_query_2.replace('{wd_string}', ws_list_cs_2)
sql_df2 = ae.run_query(conn, sql_queryf2)


df_exploded = price_change_df_send2[['fs_ticketid', 'fs_uk', 'Worksheet_Details']].explode('Worksheet_Details')
df_exploded = (
    df_exploded
    .dropna(subset=['Worksheet_Details'])   # Remove None/NaN rows
    .assign(Worksheet_Details=lambda x: x['Worksheet_Details'].astype(str).str.strip()) # Strip whitespace
    .loc[lambda x: x['Worksheet_Details'].str.len() > 3] # Keep only if length > 3
)
df_exploded = df_exploded.reset_index(drop=True)

sql_df2['CS_COST_CHANGE'] = (pd.to_numeric(sql_df2['CS_COST_CHANGE'], errors='coerce').fillna(0).astype(int).astype(str))
sql_full_cs_exp = sql_df2.merge(df_exploded,how='left',right_on='Worksheet_Details',left_on='CS_COST_CHANGE')
sql_full_cs_exp['cs_sql_ticketid'] = sql_full_cs_exp['fs_ticketid'].apply(ae.utils.force_integer_format)
sql_full_cs_exp['CS_DEPT_NO'] = sql_full_cs_exp['CS_DEPT_NO'].apply(ae.utils.force_integer_format)
sql_full_cs_exp['CS_FASHION_STYLE_NO'] = sql_full_cs_exp['CS_FASHION_STYLE_NO'].apply(ae.utils.force_integer_format)
sql_full_cs_exp['CS_ITEM_ID'] = sql_full_cs_exp['CS_ITEM_ID'].apply(ae.utils.force_integer_format)
sql_full_cs_exp['CS_STATUS'] = sql_full_cs_exp['CS_STATUS'].str.strip().str.upper()
sql_full_cs_exp['cs_sql_shape'] = sql_full_cs_exp.groupby('cs_sql_ticketid')['cs_sql_ticketid'].transform('count')
sql_full_cs_exp['cs_sql_fhs_count'] = (sql_full_cs_exp['CS_FASHION_STYLE_NO'].astype(str) + '_' + sql_full_cs_exp['CS_DEPT_NO'].astype(str) + '_' + sql_full_cs_exp['CS_MFG_NO'].astype(str)).groupby(sql_full_cs_exp['cs_sql_ticketid']).transform('nunique')
sql_full_cs_exp['cs_sql_fhs_ws_count'] = sql_full_cs_exp.groupby(['cs_sql_ticketid','CS_DEPT_NO','CS_MFG_NO'])['CS_FASHION_STYLE_NO'].transform('nunique')
sql_full_cs_exp['cs_sql_SELF_PROD_ID2'] = '_'+sql_full_cs_exp['CS_FASHION_STYLE_NO'].astype(str)+'_'+sql_full_cs_exp['CS_ITEM_ID'].astype(str)+'_'
cols = list(sql_full_cs_exp.columns)
cols.insert(0, cols.pop(cols.index('cs_sql_ticketid')))
sql_full_cs_exp = sql_full_cs_exp[cols]
sql_full_cs = sql_full_cs_exp[ae.config.sql_cs_req_cols]

sql_df['WORKSHEET_NO'] = (pd.to_numeric(sql_df['WORKSHEET_NO'], errors='coerce').fillna(0).astype(int).astype(str))
sql_full = sql_df.merge(df_exploded,how='left',right_on='Worksheet_Details',left_on='WORKSHEET_NO')
sql_full['CHANGE_PCT2'] = sql_full['CHANGE_PCT'].apply(ae.utils.normalize_pct).round(2)
sql_full['sql_ticketid'] = sql_full['fs_ticketid'].apply(ae.utils.force_integer_format)
sql_full['sql_unique_price_type_code_list'] = sql_full.groupby('sql_ticketid')['PRICE_TYPE_CODE'].transform(lambda x: [x.unique().tolist()] * len(x))
sql_full['sql_unique_special_ins_list'] = sql_full.groupby('sql_ticketid')['SPECIAL_INSTRUCTIONS'].transform(lambda x: [x.unique().tolist()] * len(x))
sql_full['sql_unique_req_reason_no_list'] = sql_full.groupby('sql_ticketid')['REQ_REASON_NO'].transform(lambda x: [x.unique().tolist()] * len(x))
sql_full['FASHION_STYLE_NUMBER'] = sql_full['FASHION_STYLE_NUMBER'].apply(ae.utils.force_integer_format)
sql_full['SKN_NO'] = sql_full['SKN_NO'].apply(ae.utils.force_integer_format)
sql_full['DEPARTMENT_NUMBER'] = sql_full['DEPARTMENT_NUMBER'].apply(ae.utils.force_integer_format)
sql_full['sql_shape'] = sql_full.groupby('sql_ticketid')['sql_ticketid'].transform('count')
sql_full['sql_unique_change_pct_list'] = sql_full.groupby('sql_ticketid')['CHANGE_PCT2'].transform(lambda x: [x.unique().tolist()] * len(x))
sql_full['sql_unique_status_list'] = sql_full.groupby('sql_ticketid')['STATUS'].transform(lambda x: [x.unique().tolist()] * len(x))
sql_full['sql_unique_price_type_code_list'] = sql_full.groupby('sql_ticketid')['PRICE_TYPE_CODE'].transform(lambda x: [x.unique().tolist()] * len(x))
sql_full['sql_unique_special_ins_list'] = sql_full.groupby('sql_ticketid')['SPECIAL_INSTRUCTIONS'].transform(lambda x: [x.unique().tolist()] * len(x))
sql_full['sql_unique_req_reason_no_list'] = sql_full.groupby('sql_ticketid')['REQ_REASON_NO'].transform(lambda x: [x.unique().tolist()] * len(x))
sql_full['sql_fhs_count'] = (sql_full['FASHION_STYLE_NUMBER'].astype(str) + '_' + sql_full['DEPARTMENT_NUMBER'].astype(str) + '_' + sql_full['MANUFACTURER_NUMBER'].astype(str)).groupby(sql_full['sql_ticketid']).transform('nunique')
sql_full['sql_fhs_ws_count'] = sql_full.groupby(['sql_ticketid','DEPARTMENT_NUMBER','MANUFACTURER_NUMBER'])['FASHION_STYLE_NUMBER'].transform('nunique')
sql_full['sql_SELF_PROD_ID2'] = '_'+sql_full['FASHION_STYLE_NUMBER'].astype(str)+'_'+sql_full['SKN_NO'].astype(str)+'_'
cols = list(sql_full.columns)
cols.insert(0, cols.pop(cols.index('sql_ticketid')))
sql_full = sql_full[cols]

attach_full = ae.utils.merge_xlsx_files('Fixed_attachments',f'data_reports/Attachments_Full_Export_{ts_prefix}.xlsx')
attach_full = attach_full[["DEPARTMENT_NUMBER","MANUFACTURER_NUMBER","FASHION_STYLE_NUMBER","STATIC_SKN","UPC","CHANGE_PERC","NEW_RETAIL_PRICE","NEW_COST_PRICE","source_file","filled_cols"]]
attach_full['ticket_id'] = (attach_full['source_file'].fillna('').str.replace('fixed_', '', case=False).str.split('_', n=2, expand=True)[1].fillna('error_no_id')).astype(int)
score_array = np.where(attach_full['source_file'].str.contains('merged', case=False, na=False), 0, 1)
score_series = pd.Series(score_array, index=attach_full.index)
best_score = score_series.groupby(attach_full['ticket_id']).transform('min')
attach_full = attach_full[score_series == best_score]
attach_full['FASHION_STYLE_NUMBER'] = attach_full['FASHION_STYLE_NUMBER'].apply(ae.utils.force_integer_format)
attach_full['STATIC_SKN'] = attach_full['STATIC_SKN'].apply(ae.utils.force_integer_format)
attach_full['UPC'] = attach_full['UPC'].apply(ae.utils.force_integer_format)
attach_full = attach_full[~attach_full['FASHION_STYLE_NUMBER'].astype(str).str.contains(r'^9+(?:\.0+)?$', na=False)].reset_index(drop=True)
attach_full = attach_full.dropna(thresh=2).reset_index(drop=True) # Will delete row if <2 columns have info/values
attach_full['SELF_PROD_ID2'] = attach_full.apply(ae.core.build_conc_id, axis=1)
attach_full[~attach_full['SELF_PROD_ID2'].str.len()<7].reset_index(drop=True,inplace=True)
attach_full['DEPARTMENT_NUMBER'] = attach_full['DEPARTMENT_NUMBER'].apply(ae.utils.force_integer_format)
attach_full['attach_shape'] = attach_full.groupby('ticket_id')['ticket_id'].transform('count')
attach_full['CHANGE_PERC'] = attach_full['CHANGE_PERC'].apply(ae.utils.normalize_pct).round(2)
attach_full['NEW_COST_PRICE'] = attach_full['NEW_COST_PRICE'].apply(ae.utils.force_float_format)
attach_full['NEW_RETAIL_PRICE'] = attach_full['NEW_RETAIL_PRICE'].apply(ae.utils.force_float_format_v2)
grouped_series = (attach_full.groupby('ticket_id')['CHANGE_PERC'].apply(lambda x: (clean := x.map(ae.utils.normalize_pct).dropna().round(2).unique().tolist(),clean if len(clean) > 0 else np.nan)[1]))
attach_full['attach_perc_list'] = attach_full['ticket_id'].map(grouped_series)

cols = list(attach_full.columns)
cols.insert(0, cols.pop(cols.index('ticket_id')))
attach_full = attach_full[cols]

Found 21 files. Starting merge...


In [15]:
fs_attach_df = duckdb.query(ae.fs_attach_merge_q).to_df()
# fs_attach_sql_df = duckdb.query(ae.fs_attach_sql_merge_q).to_df()
fs_attach_sql_df = duckdb.query(ae.fs_attach_sql_pcw_cs_merge_q).to_df()
fs_attach_sql_df.columns = fs_attach_sql_df.columns.str.lower()
audit_df1 = duckdb.query(ae.audit_sql_1).to_df()
audit_df2 = duckdb.query(ae.audit_sql_2).to_df()
audit_df3 = duckdb.query(ae.audit_sql_3).to_df()
audit_df4 = duckdb.query(ae.audit_sql_4).to_df()

In [16]:
if audit_df3["fs_ticketid"].duplicated().any():
    # Find the specific duplicates to help you debug
    duplicates = audit_df3[audit_df3["fs_ticketid"].duplicated()]["fs_ticketid"].unique()
    raise ValueError(f"CRITICAL: Duplicates found in fs_ticketid: {list(duplicates)}")

In [17]:
temp_audit3 =  audit_df3[ae.utils.union_lists_multiple([ae.config.audit_common_cols,ae.config.audit3_full_check_cols,ae.config.audit_summary_cols])]
# temp_audit4 =  audit_df4[ae.utils.union_lists_multiple([ae.config.audit_common_cols,ae.config.audit3_full_check_cols,ae.config.audit_summary_cols])]
temp_audit4 =  audit_df4
temp_audit2_man = audit_df2[ae.config.sql_audit2_req_cols]
temp_audit2 = audit_df2

for enm, (idx, r) in enumerate(temp_audit3.iterrows()):
    # if r['fs_ticketid'] == 122284:
    if 1==1:
        pass
    else:
        continue
    if r['fs_price_change_type_cleaned']=='perm markdown' and r['fs_markdown_code']==24:
        exclude_cols = ae.config.audit_penny_exclude
    elif r['fs_price_change_type_cleaned']=='perm markdown' and r['fs_markdown_code']!=24:
        exclude_cols = ae.config.audit_perm_mkdn_exclude
    elif r['fs_price_change_type_cleaned']=='regular / regular':
        exclude_cols = ae.config.audit_reg_exclude
    elif r['fs_price_change_type_cleaned']=='markdown cancel':
        exclude_cols = ae.config.audit_cancel_exclude
    check_cols_temp = ae.utils.subtract_lists(ae.config.audit3_full_check_cols,exclude_cols)
    audit_cols = ae.utils.union_lists_multiple([ae.config.audit_common_cols,check_cols_temp,ae.config.audit_summary_cols])
    temp_audit3.loc[idx, exclude_cols] = 0
    ticket_data = temp_audit2[temp_audit2['fs_ticketid'] == r['fs_ticketid']]
    for col in check_cols_temp:
        if pd.notna(r[col]) and r[col] == 1:
        # if (pd.notna(r[col]) and r[col] == 1)&(r['fs_ticketid'] == 119733)&(col=='chk_reg_price_missing_tkt_flag'):
            temp_col = col.replace('mismatch_tkt_flag','match_flag').replace('missing_tkt_flag','missing_flag')
            temp_col2 = col.replace('tkt_flag','summary')
            # print(col,temp_col,temp_col2)
            f_temp_audit2 = ticket_data[(ticket_data[temp_col]==0)]
            
            if 'missing' in col:
                dict_builder = ticket_data[(ticket_data['fs_ticketid']==r['fs_ticketid'])]
            else:
                dict_builder = f_temp_audit2
            if '_cs_' in col:
                mfg_dept_ws_dict = (dict_builder[['sql_cs_department_number','sql_cs_manufacturer_number', 'sql_cs_worksheet_no']].dropna().drop_duplicates().groupby(['sql_cs_department_number','sql_cs_manufacturer_number'])['sql_cs_worksheet_no'].apply(lambda x: ", ".join(x.astype(int).astype(str))).to_dict())
            else:
                mfg_dept_ws_dict = (dict_builder[['sql_department_number','sql_manufacturer_number', 'sql_worksheet_no']].dropna().drop_duplicates().groupby(['sql_department_number','sql_manufacturer_number'])['sql_worksheet_no'].apply(lambda x: ", ".join(x.astype(int).astype(str))).to_dict())
            summary_text = ae.core.summary_txt_builder_v6(f_temp_audit2,mfg_dept_ws_dict,col)
            # if col == 'chk_reg_price_missing_tkt_flag':
                # print(col,mfg_dept_ws_dict,'----------',summary_text)
                # k = f_temp_audit2
            temp_audit3.at[idx, temp_col2] = summary_text

temp_audit3[ae.config.audit3_full_check_cols] = (temp_audit3[ae.config.audit3_full_check_cols]==0)
temp_audit3.loc[temp_audit3['fs_cost_updates_req_flag'] == False, 'chk_cs_dol_missing_tkt_flag'] = True
temp_audit3.loc[temp_audit3['fs_cost_updates_req_flag'] == False, 'chk_cs_dol_missing_summary'] = ''

temp_audit4[ae.config.audit3_full_check_cols]=True
# audit_df_final_pre = pd.concat([temp_audit3,temp_audit4])

audit_df_final_pre = temp_audit3
audit_df_final_pre[ae.config.list_clean_cols] = audit_df_final_pre[ae.config.list_clean_cols].apply(lambda col: col.apply(ae.utils.clean_concat_logic))
audit_df_final_pre.drop(columns=['chk_fhs_count_mismatch_summary'], inplace=True)
audit_df_final_pre['Blank_1'] = ''
audit_df_final_pre['Blank_2'] = ''

temp_audit4[ae.config.list_clean_cols] = temp_audit4[ae.config.list_clean_cols].apply(lambda col: col.apply(ae.utils.clean_concat_logic))
temp_audit4.drop(columns=['chk_fhs_count_mismatch_summary'], inplace=True)
temp_audit4['Blank_1'] = ''
temp_audit4['Blank_2'] = ''

In [18]:
audit_df_final = pd.DataFrame(audit_df_final_pre)
audit_df_final = audit_df_final.reset_index(drop=True)
audit_df_final.loc[audit_df_final['fs_markdown_code'] == 21, ['chk_perc_match_flag']] = True
flag_cols = ae.config.audit3_full_check_cols
audit_df_final['chk_all_flags_match'] = audit_df_final[flag_cols].all(axis=1)
audit_df_final['chk_all_flags_match'] = audit_df_final['chk_all_flags_match'].fillna(False)
audit_df_final['flags_count_matched'] = audit_df_final[flag_cols].sum(axis=1)
audit_df_final['flags_count_failed'] = len(flag_cols) - audit_df_final['flags_count_matched']
audit_df_final['failed_reasons'] = audit_df_final[flag_cols].apply(lambda x: ' | '.join([col.replace('chk_', '').replace('_match', '').replace('_', ' ').strip().upper() for col in x.index[~x.fillna(False).astype(bool)]]) if not x.fillna(False).all() else 'PASS', axis=1)
audit_df_final['pass_reasons'] = audit_df_final[flag_cols].apply(lambda x: ' | '.join([col.replace('chk_', '').replace('_match', '').replace('_', ' ').strip().upper() for col in x.index[x.fillna(False).astype(bool)]]) if x.fillna(False).any() else 'NONE', axis=1)

# pcols = ae.config.audit_df_final_col_order
# audit_df_final = audit_df_final[pcols + audit_df_final.columns.difference(pcols).tolist()]

audit_df_final[flag_cols] = audit_df_final[flag_cols].fillna(True)
audit_df_final = audit_df_final[ae.config.audit_df_final_col_order]
audit_df_final.columns = ae.config.audit_df_final_col_order_rnm

In [19]:
wrap_targets = ['sql_unique_special_ins_list','attach_perc_list','sql_unique_change_pct_list','chk_sql_ws_status_mismatch_summary',
       'chk_effective_date_mismatch_summary',
       'chk_markdown_code_mismatch_summary',
       'chk_price_type_code_mismatch_summary',
       'chk_fhs_dir_mismatch_summary', 'chk_perc_mismatch_summary',
       'chk_reg_price_mismatch_summary', 'chk_cancel_price_mismatch_summary',
       'chk_penny_price_mismatch_summary', 'chk_reg_price_missing_summary',
       'chk_all_missing_summary','chk_cs_active_date_mismatch_summary',
       'chk_cs_fhs_count_mismatch_summary','chk_cs_status_mismatch_summary','chk_cs_costcng_mismatch_summary'
                ,'chk_cs_dol_missing_summary'] # Add more column names to this list as needed
color_targets = [c for c in audit_df_final.columns if 'chk_' in c or 'match' in c]
# ---------------------------

# 2. Logic Functions
def apply_excel_styles(val):
    style = ''
    if val is False:
        style += 'background-color: #FFC7CE; color: #9C0006;'
    elif val is True:
        style += 'background-color: #C6EFCE; color: #006100;'
    return style

def force_wrap(val):
    return 'white-space: pre-wrap;'

# 3. Apply Styling via Pandas Styler
styled_audit_df = audit_df_final.style.applymap(
    apply_excel_styles, subset=color_targets
).applymap(
    force_wrap, subset=wrap_targets
)

# 4. Export Setup
output_fn = f"data_reports/Final_Audit_Results_{ts_prefix}.xlsx"
writer = pd.ExcelWriter(output_fn, engine='xlsxwriter')
styled_audit_df.to_excel(writer, index=False, sheet_name='Audit Summary')

workbook  = writer.book
worksheet = writer.sheets['Audit Summary']

# 5. Global Formatting
normal_format = workbook.add_format({'valign': 'vcenter'})
# Set all columns to standard width (1x)
worksheet.set_column(0, len(audit_df_final.columns) - 1, 15, normal_format)

# 6. Apply List-based Width and Height logic
# Set the wide width for everything in your wrap list
for col_name in wrap_targets:
    if col_name in audit_df_final.columns:
        idx = audit_df_final.columns.get_loc(col_name)
        worksheet.set_column(idx, idx, 50) # 3x-ish Width

# Set all rows to 3x height (60 units)
for row_num in range(1, len(audit_df_final) + 1):
    worksheet.set_row(row_num, 60)

# 6.5 Convert the data range into an Excel Table
(max_row, max_col) = audit_df_final.shape
column_settings = [{'header': column} for column in audit_df_final.columns]

worksheet.add_table(0, 0, max_row, max_col - 1, {
    'columns': column_settings,
    'style': 'Table Style Medium 9', # You can change the number for different colors
    'name': 'AuditTable'
})
# Add SQL Full Raw Data

if 'temp_audit4' in locals() and not temp_audit4.empty:
    temp_audit4.to_excel(writer, index=False, sheet_name='No data')

if 'attach_full' in locals() and not attach_full.empty:
    attach_full.to_excel(writer, index=False, sheet_name='Attach Raw Data')

if 'sql_full' in locals() and not sql_full.empty:
    sql_full.to_excel(writer, index=False, sheet_name='SQL PCW Raw Data')

if 'sql_full_cs_exp' in locals() and not sql_full_cs_exp.empty:
    sql_full_cs_exp.to_excel(writer, index=False, sheet_name='SQL CS Raw Data')

if 'issue_files' in locals() and not issue_files.empty:
    issue_files.to_excel(writer, index=False, sheet_name='Issue Files')
# 7. Close and Save
writer.close()

print(f"\n" + "="*100)
print(f"✅ Report Fixed & Generated: {output_fn}")
print(f"📝 Wrapped Columns: {wrap_targets}")
print(f"📊 Colored Columns: {len(color_targets)} identified")
print("="*100)
print(f"Ended at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

C:\Users\MBathija\AppData\Local\Temp\ipykernel_12376\1396983309.py:27: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  styled_audit_df = audit_df_final.style.applymap(
C:\Users\MBathija\AppData\Local\Temp\ipykernel_12376\1396983309.py:29: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  ).applymap(



✅ Report Fixed & Generated: data_reports/Final_Audit_Results_202603162019.xlsx
📝 Wrapped Columns: ['sql_unique_special_ins_list', 'attach_perc_list', 'sql_unique_change_pct_list', 'chk_sql_ws_status_mismatch_summary', 'chk_effective_date_mismatch_summary', 'chk_markdown_code_mismatch_summary', 'chk_price_type_code_mismatch_summary', 'chk_fhs_dir_mismatch_summary', 'chk_perc_mismatch_summary', 'chk_reg_price_mismatch_summary', 'chk_cancel_price_mismatch_summary', 'chk_penny_price_mismatch_summary', 'chk_reg_price_missing_summary', 'chk_all_missing_summary', 'chk_cs_active_date_mismatch_summary', 'chk_cs_fhs_count_mismatch_summary', 'chk_cs_status_mismatch_summary', 'chk_cs_costcng_mismatch_summary', 'chk_cs_dol_missing_summary']
📊 Colored Columns: 39 identified
Ended at 2026-03-16 20:24:09


In [24]:
[]==False

False